# El pingüino promedio no vino a la junta

**Nivel:** principiante

[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jcval94/narrative/blob/codex/corporate-data-narrative-lab/corporate-data-narrative-lab/outputs/notebooks/05-el-pinguino-promedio-no-vino.ipynb)

## Pregunta central

¿Cómo resumimos, comparamos y revisamos los registros antes de publicar una cifra o descartar un caso raro?

## Recreación narrativa

> **Marta:** "El cartel dirá: ‘Un pingüino pesa...’ y ponemos una cifra."
>
> **Nadia:** "¿Cuál pingüino?"
>
> **Marta:** "El promedio."
>
> **Tomás:** "Perfecto. ¿Le ponemos especie?"
>
> **Marta:** "No cabe."
>
> **Nadia:** "Entonces el dato cabe y el pingüino no."

*La escena es una recreación; las conclusiones provienen del dataset citado.*

## Fuente real

**Palmer Archipelago (Antarctica) penguin data — penguins.csv**, Palmer Station LTER; paquete palmerpenguins de Horst, Hill y Gorman. [Página de origen](https://allisonhorst.github.io/palmerpenguins/) · [datos](https://raw.githubusercontent.com/allisonhorst/palmerpenguins/main/inst/extdata/penguins.csv) · licencia: CC0 1.0 Universal.  
Consultado: 2026-07-18 · 344 filas · columnas usadas: `species`, `island`, `bill_length_mm`, `bill_depth_mm`, `flipper_length_mm`, `body_mass_g`, `sex`, `year`.

In [1]:
# @title Preparar los datos { display-mode: "form" }
especie = "Todas" # @param ["Todas", "Adelie", "Chinstrap", "Gentoo"]
bins = 18 # @param {type:"slider", min:6, max:40, step:1}
suavizado = 0.65 # @param {type:"slider", min:0.4, max:1.2, step:0.05}
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from IPython.display import Markdown, display
DATA_URL = "https://raw.githubusercontent.com/allisonhorst/palmerpenguins/main/inst/extdata/penguins.csv"
df_completo = pd.read_csv(DATA_URL)
df = df_completo if especie == "Todas" else df_completo.query("species == @especie")
df = df.reset_index(drop=True)
n_masas = int(df["body_mass_g"].notna().sum())
display(Markdown(f"**Vista:** {especie} · **{len(df)} registros** · **{n_masas} masas válidas** · **bins={bins}** · **KDE={suavizado}**"))
df[["species", "island", "body_mass_g", "flipper_length_mm", "sex", "year"]].head()

**Vista:** Todas · **344 registros** · **342 masas válidas** · **bins=18** · **KDE=0.65**

,species,island,body_mass_g,flipper_length_mm,sex,year
0,Adelie,Torgersen,3750.0,181.0,male,2007
1,Adelie,Torgersen,3800.0,186.0,female,2007
2,Adelie,Torgersen,3250.0,195.0,female,2007
3,Adelie,Torgersen,NaN,NaN,NaN,2007
4,Adelie,Torgersen,3450.0,193.0,female,2007


## 1. Resumen numérico

> **Tomás:** "Dame el número correcto."
>
> **Nadia:** "Hay varios correctos; contestan preguntas distintas."
>
> **Marta:** "Dame el que ocupe menos caracteres."


**Pregunta:** ¿Qué cuentan media, mediana, moda, rango, varianza, desviación y percentiles sobre la misma masa?

**Conexión:** Punto de partida: una cifra cómoda quiere representar toda la tabla.

In [2]:
masa = df["body_mass_g"].dropna()
modas = masa.mode().tolist()
resumen = pd.Series({
    "n": len(masa), "media": masa.mean(), "mediana": masa.median(), "moda(s)": modas,
    "rango": masa.max() - masa.min(), "varianza muestral": masa.var(ddof=1),
    "desviación estándar muestral": masa.std(ddof=1),
    "p10 · p25 · p75 · p90": masa.quantile([.10, .25, .75, .90]).round(1).to_dict()})
resumen

n                                                                             342
media                                                                 4201.754386
mediana                                                                    4050.0
moda(s)                                                                  [3800.0]
rango                                                                      3600.0
varianza muestral                                                   643131.077327
desviación estándar muestral                                           801.954536
p10 · p25 · p75 · p90           {0.1: 3300.0, 0.25: 3550.0, 0.75: 4750.0, 0.9:...
dtype: object

In [3]:
# @title Explorar Resumen numérico { display-mode: "form" }
grupo = especie
q10, q25, q75, q90 = masa.quantile([.10, .25, .75, .90])
media, mediana = resumen["media"], resumen["mediana"]
fig = go.Figure()
fig.add_trace(go.Violin(x=masa, y=[grupo] * len(masa), orientation="h", name="Distribución", box_visible=False, meanline_visible=False, points="all", jitter=.22, pointpos=-1.25, fillcolor="#DCE8E6", line_color="#2F5D62", marker={"color": "#2F5D62", "size": 4, "opacity": .28}, hovertemplate="masa=%{x:,.0f} g<extra></extra>"))
fig.add_trace(go.Scatter(x=[masa.min(), masa.max()], y=[grupo, grupo], mode="lines", name="Rango", line={"color": "#173F5F", "width": 2}, hovertemplate="extremo=%{x:,.0f} g<extra>rango</extra>"))
fig.add_trace(go.Scatter(x=[q10, q90], y=[grupo, grupo], mode="lines+markers", name="p10–p90", line={"color": "#C86B3C", "width": 6}, marker={"color": "#C86B3C", "size": 8}, hovertemplate="percentil=%{x:,.0f} g<extra>p10–p90</extra>"))
fig.add_trace(go.Scatter(x=[q25, q75], y=[grupo, grupo], mode="lines+markers", name="p25–p75", line={"color": "#D99B2B", "width": 12}, marker={"color": "#D99B2B", "size": 10}, hovertemplate="cuartil=%{x:,.0f} g<extra>p25–p75</extra>"))
fig.add_trace(go.Scatter(x=[mediana], y=[grupo], mode="markers", name="Mediana", marker={"color": "#173F5F", "size": 14, "symbol": "circle-open", "line": {"width": 3}}, hovertemplate="mediana=%{x:,.0f} g<extra></extra>"))
fig.add_trace(go.Scatter(x=[media], y=[grupo], mode="markers", name="Media", marker={"color": "#C84B74", "size": 14, "symbol": "diamond"}, hovertemplate="media=%{x:,.1f} g<extra></extra>"))
fig.add_trace(go.Scatter(x=modas, y=[grupo] * len(modas), mode="markers", name="Moda(s)", marker={"color": "#6E5AA8", "size": 13, "symbol": "star"}, hovertemplate="moda=%{x:,.0f} g<extra></extra>"))
todos = [True] * 7
fig.update_layout(template="plotly_white", height=470, title=f"Resumen de la masa corporal<br><sup>{grupo} · n={len(masa)} · varianza y desviación muestrales</sup>", xaxis_title="Masa corporal (g)", yaxis_title="Distribución seleccionada", legend={"orientation": "h", "y": -0.22}, margin={"t": 105, "b": 110}, font={"family": "Arial", "color": "#173F5F"}, hoverlabel={"bgcolor": "#FFFFFF"}, updatemenus=[{"x": 1, "y": 1.19, "xanchor": "right", "buttons": [{"label": "Todo", "method": "update", "args": [{"visible": todos}]}, {"label": "Centro", "method": "update", "args": [{"visible": [True, False, False, False, True, True, True]}]}, {"label": "Dispersión", "method": "update", "args": [{"visible": [True, True, True, True, False, False, False]}]}]}])
fig.show()


display(Markdown("**Lo que muestra:** La media y la mediana ubican el centro; la moda señala el valor más repetido. Rango, varianza y desviación cuantifican dispersión, mientras p10–p90 muestra dónde cae el 80% central sin fingir que todos pesan igual."))

**Lo que muestra:** La media y la mediana ubican el centro; la moda señala el valor más repetido. Rango, varianza y desviación cuantifican dispersión, mientras p10–p90 muestra dónde cae el 80% central sin fingir que todos pesan igual.

## 2. Distribuciones

> **Nadia:** "La gráfica tiene más de un montón."
>
> **Tomás:** "¿Son dos tipos de promedio?"
>
> **Marta:** "No. Son los pingüinos reclamando su especie."


**Pregunta:** ¿Qué revelan histograma, densidad, sesgo, picos y elección de bins que el resumen no puede mostrar?

**Conexión:** Resumen numérico describió centro y dispersión, pero todavía no mostró la forma ni explicó por qué una sola cifra mezcla especies.

In [4]:
masa = df["body_mass_g"].dropna()
grid = np.linspace(masa.min(), masa.max(), 240)
ancho = suavizado * 1.06 * masa.std() * len(masa) ** (-1 / 5)
densidad = np.exp(-.5 * ((grid[:, None] - masa.to_numpy()) / ancho) ** 2).mean(1) / (ancho * np.sqrt(2 * np.pi))
picos = grid[1:-1][(densidad[1:-1] > densidad[:-2]) & (densidad[1:-1] >= densidad[2:])]
forma = pd.Series({"bins": bins, "suavizado KDE": suavizado, "sesgo": masa.skew(), "picos KDE": len(picos),
                   "ubicación de picos (g)": np.round(picos).astype(int).tolist()})
forma

bins                                      18
suavizado KDE                           0.65
sesgo                               0.470329
picos KDE                                  3
ubicación de picos (g)    [3649, 4643, 5426]
dtype: object

In [5]:
# @title Explorar Distribuciones { display-mode: "form" }
paleta = {"Adelie": "#2F5D62", "Chinstrap": "#C86B3C", "Gentoo": "#C84B74"}
fig = go.Figure()
grupos = list(df.dropna(subset=["body_mass_g"]).groupby("species", observed=True))
for nombre, datos in grupos:
    fig.add_trace(go.Histogram(x=datos["body_mass_g"], nbinsx=bins, histnorm="probability density", name=nombre, opacity=.48, marker={"color": paleta[nombre], "line": {"color": "#FFFFFF", "width": .7}}, hovertemplate=f"{nombre}<br>masa=%{{x:,.0f}} g<br>densidad=%{{y:.5f}}<extra></extra>"))
fig.add_trace(go.Scatter(x=grid, y=densidad, mode="lines", name="Densidad KDE", line={"color": "#D99B2B", "width": 4}, fill="tozeroy", fillcolor="rgba(217,155,43,.10)", hovertemplate="masa=%{x:,.0f} g<br>densidad=%{y:.5f}<extra>KDE</extra>"))
fig.add_trace(go.Scatter(x=picos, y=np.interp(picos, grid, densidad), mode="markers+text", text=[f"pico {i+1}" for i in range(len(picos))], textposition="top center", name="Picos KDE", marker={"color": "#6E5AA8", "size": 11, "symbol": "diamond"}, hovertemplate="pico=%{x:,.0f} g<extra></extra>"))
n_hist = len(grupos)
fig.add_vline(x=resumen["media"], line={"color": "#C84B74", "width": 2, "dash": "dash"}, annotation_text="media", annotation_position="top right")
fig.add_vline(x=resumen["mediana"], line={"color": "#173F5F", "width": 2, "dash": "dot"}, annotation_text="mediana", annotation_position="top left")
fig.update_layout(template="plotly_white", barmode="overlay", height=520, title=f"Histograma y densidad de la masa corporal<br><sup>{especie} · n={len(masa)} · {bins} bins · KDE={suavizado} · sesgo={masa.skew():.2f}</sup>", xaxis_title="Masa corporal (g)", yaxis_title="Densidad de probabilidad", legend={"orientation": "h", "y": -0.22}, margin={"t": 110, "b": 115}, font={"family": "Arial", "color": "#173F5F"}, hoverlabel={"bgcolor": "#FFFFFF"}, updatemenus=[{"x": 1, "y": 1.19, "xanchor": "right", "buttons": [{"label": "Histograma + densidad", "method": "update", "args": [{"visible": [True] * (n_hist + 2)}]}, {"label": "Sólo histograma", "method": "update", "args": [{"visible": [True] * n_hist + [False, False]}]}, {"label": "Sólo densidad", "method": "update", "args": [{"visible": [False] * n_hist + [True, True]}]}]}])
fig.show()


display(Markdown("**Lo que muestra:** Con «Todas» y el suavizado inicial aparecen varios picos; el color por especie explica la mezcla. Cambiar bins o KDE altera la forma aparente, no las masas: usa ambos controles y el filtro para comprobar si un pico es estable."))

**Lo que muestra:** Con «Todas» y el suavizado inicial aparecen varios picos; el color por especie explica la mezcla. Cambiar bins o KDE altera la forma aparente, no las masas: usa ambos controles y el filtro para comprobar si un pico es estable.

## 3. Comparación visual

> **Marta:** "Pon barras; se ven ejecutivas."
>
> **Nadia:** "Las barras esconden la forma que acabamos de encontrar."
>
> **Tomás:** "Entonces ponemos cuatro gráficas."
>
> **Marta:** "Había un hueco, no una exposición."


**Pregunta:** ¿Qué conserva y qué oculta cada opción: barras, boxplot, violin plot y ECDF?

**Conexión:** Distribuciones reveló forma y mezcla; ahora debemos elegir qué comparación conservará esa evidencia en el cartel.

In [6]:
comparacion = df.groupby("species")["body_mass_g"].agg(
    n="count", media="mean", mediana="median", desviacion="std",
    minimo="min", maximo="max")
comparacion.round(1)

,n,media,mediana,desviacion,minimo,maximo
species,,,,,,
Adelie,151,3700.7,3700.0,458.6,2850.0,4775.0
Chinstrap,68,3733.1,3700.0,384.3,2700.0,4800.0
Gentoo,123,5076.0,5000.0,504.1,3950.0,6300.0


In [7]:
# @title Explorar Comparación visual { display-mode: "form" }
especies = comparacion.index.tolist()
colores = [paleta[nombre] for nombre in especies]
fig = go.Figure()
fig.add_trace(go.Bar(x=especies, y=comparacion["media"], text=comparacion["media"].round(0), customdata=comparacion[["n", "mediana", "desviacion"]], marker={"color": colores, "line": {"color": "#173F5F", "width": 1}}, name="Media", hovertemplate="%{x}<br>media=%{y:,.0f} g<br>mediana=%{customdata[1]:,.0f} g<br>n=%{customdata[0]}<extra></extra>"))
for nombre in especies:
    valores = df.loc[df["species"] == nombre, "body_mass_g"].dropna()
    fig.add_trace(go.Box(x=[nombre] * len(valores), y=valores, name=nombre, marker_color=paleta[nombre], line_color=paleta[nombre], boxpoints="outliers", visible=False, showlegend=False, hovertemplate=f"{nombre}<br>masa=%{{y:,.0f}} g<extra>boxplot</extra>"))
for nombre in especies:
    valores = df.loc[df["species"] == nombre, "body_mass_g"].dropna()
    fig.add_trace(go.Violin(x=[nombre] * len(valores), y=valores, name=nombre, fillcolor=paleta[nombre], line_color="#173F5F", opacity=.65, box_visible=True, meanline_visible=True, points=False, visible=False, showlegend=False, hovertemplate=f"{nombre}<br>masa=%{{y:,.0f}} g<extra>violín</extra>"))
for nombre in especies:
    orden = np.sort(df.loc[df["species"] == nombre, "body_mass_g"].dropna())
    fig.add_trace(go.Scatter(x=orden, y=np.arange(1, len(orden)+1)/len(orden), mode="lines", name=nombre, line={"color": paleta[nombre], "width": 3}, visible=False, hovertemplate=f"{nombre}<br>masa≤%{{x:,.0f}} g<br>proporción=%{{y:.1%}}<extra>ECDF</extra>"))
n = len(especies); total = 1 + 3 * n
barra = [True] + [False] * (total - 1); caja = [False] + [True] * n + [False] * (2*n); violin = [False] * (1+n) + [True] * n + [False] * n; ecdf = [False] * (1+2*n) + [True] * n
base_titulo = f"Comparación visual de masa por especie<br><sup>{especie} · n={int(comparacion['n'].sum())} masas válidas"
botones = [{"label": "Barras", "method": "update", "args": [{"visible": barra}, {"title.text": base_titulo + " · media</sup>", "xaxis.title.text": "Especie", "xaxis.type": "category", "yaxis.title.text": "Masa corporal media (g)", "yaxis.range": [0, 6500], "showlegend": False}]}, {"label": "Boxplot", "method": "update", "args": [{"visible": caja}, {"title.text": base_titulo + " · mediana, IQR y extremos</sup>", "xaxis.title.text": "Especie", "xaxis.type": "category", "yaxis.title.text": "Masa corporal (g)", "yaxis.range": [2500, 6500], "showlegend": False}]}, {"label": "Violin plot", "method": "update", "args": [{"visible": violin}, {"title.text": base_titulo + " · forma y densidad</sup>", "xaxis.title.text": "Especie", "xaxis.type": "category", "yaxis.title.text": "Masa corporal (g)", "yaxis.range": [2500, 6500], "showlegend": False}]}, {"label": "ECDF", "method": "update", "args": [{"visible": ecdf}, {"title.text": base_titulo + " · proporción acumulada</sup>", "xaxis.title.text": "Masa corporal (g)", "xaxis.type": "linear", "yaxis.title.text": "Proporción acumulada", "yaxis.range": [0, 1.02], "showlegend": True}]}]
fig.update_layout(template="plotly_white", height=520, title=base_titulo + " · media</sup>", xaxis_title="Especie", yaxis_title="Masa corporal media (g)", yaxis_range=[0, 6500], legend={"orientation": "h", "y": -0.2}, margin={"t": 110, "b": 105}, font={"family": "Arial", "color": "#173F5F"}, hoverlabel={"bgcolor": "#FFFFFF"}, updatemenus=[{"x": 1, "y": 1.19, "xanchor": "right", "buttons": botones}])
fig.show()


display(Markdown("**Lo que muestra:** Las barras comparan medias, pero borran forma y casos extremos. El boxplot compacta mediana e IQR; el violín conserva densidad; la ECDF permite leer cualquier percentil sin depender de bins. La pregunta decide la gráfica."))

**Lo que muestra:** Las barras comparan medias, pero borran forma y casos extremos. El boxplot compacta mediana e IQR; el violín conserva densidad; la ECDF permite leer cualquier percentil sin depender de bins. La pregunta decide la gráfica.

## 4. Valores atípicos

> **Tomás:** "Borro los puntos raros y la gráfica queda limpia."
>
> **Nadia:** "¿Son errores?"
>
> **Tomás:** "Son incómodos."
>
> **Marta:** "Eso es una regla de decoración, no de calidad."


**Pregunta:** ¿Un punto raro es un outlier, un caso de alto leverage, un error de captura o un caso válido que merece contexto?

**Conexión:** Comparación visual hizo visibles extremos distintos; ahora debemos decidir cuáles son outliers, cuáles tienen leverage y cuáles sugieren un problema de captura.

In [8]:
analisis = df[["species", "island", "flipper_length_mm", "body_mass_g", "sex", "year"]].copy()
analisis["error_captura"] = analisis[["body_mass_g", "flipper_length_mm"]].isna().any(axis=1) | ~analisis["body_mass_g"].between(2000, 7000) | ~analisis["flipper_length_mm"].between(150, 250)
validos = analisis.dropna(subset=["body_mass_g", "flipper_length_mm"]).copy()
q1 = validos.groupby("species")["body_mass_g"].transform(lambda s: s.quantile(.25)); q3 = validos.groupby("species")["body_mass_g"].transform(lambda s: s.quantile(.75)); iqr = q3-q1
validos["outlier_iqr"] = (validos["body_mass_g"] < q1-1.5*iqr) | (validos["body_mass_g"] > q3+1.5*iqr)
X = np.c_[np.ones(len(validos)), validos["flipper_length_mm"]]; umbral_h = 2*X.shape[1]/len(validos); validos["leverage"] = np.einsum("ij,jk,ik->i", X, np.linalg.pinv(X.T@X), X)
alto = validos["leverage"] > umbral_h; validos["tipo"] = np.select([validos["outlier_iqr"] & alto, validos["outlier_iqr"], alto], ["outlier + leverage", "raro compatible: revisar", "alto leverage"], default="común")
revision = pd.concat([analisis.query("error_captura").assign(tipo="posible error de captura"), validos.query("tipo == 'raro compatible: revisar'"), validos.query("tipo == 'outlier + leverage'"), validos.query("tipo == 'alto leverage'")]); display(revision["tipo"].value_counts()); revision.head(12)

tipo
alto leverage               18
posible error de captura     2
raro compatible: revisar     2
Name: count, dtype: int64

,species,island,flipper_length_mm,body_mass_g,sex,year,error_captura,tipo,outlier_iqr,leverage
3,Adelie,Torgersen,NaN,NaN,NaN,2007,True,posible error de captura,NaN,NaN
271,Gentoo,Biscoe,NaN,NaN,NaN,2009,True,posible error de captura,NaN,NaN
313,Chinstrap,Dream,210.0,4800.0,male,2008,False,raro compatible: revisar,True,0.004148
314,Chinstrap,Dream,192.0,2700.0,female,2008,False,raro compatible: revisar,True,0.004103
20,Adelie,Biscoe,174.0,3400.0,female,2007,False,alto leverage,False,0.013668
28,Adelie,Biscoe,172.0,3150.0,female,2007,False,alto leverage,False,0.015324
122,Adelie,Torgersen,176.0,3450.0,female,2009,False,alto leverage,False,0.012131
153,Gentoo,Biscoe,230.0,5700.0,male,2007,False,alto leverage,False,0.015470
185,Gentoo,Biscoe,230.0,6050.0,male,2007,False,alto leverage,False,0.015470
215,Gentoo,Biscoe,231.0,5650.0,male,2008,False,alto leverage,False,0.016347


In [9]:
# @title Explorar Valores atípicos { display-mode: "form" }
conteos = revision["tipo"].value_counts()
beta = np.linalg.lstsq(X, validos["body_mass_g"], rcond=None)[0]
linea_x = np.linspace(validos["flipper_length_mm"].min(), validos["flipper_length_mm"].max(), 120)
fig = go.Figure()
grupos_validos = list(validos.groupby("species", observed=True))
for nombre, datos in grupos_validos:
    fig.add_trace(go.Scatter(x=datos["flipper_length_mm"], y=datos["body_mass_g"], mode="markers", name=nombre, marker={"color": paleta[nombre], "size": 7, "opacity": .45, "line": {"color": "#FFFFFF", "width": .5}}, customdata=datos[["island", "sex", "year", "leverage"]], hovertemplate=f"{nombre}<br>aleta=%{{x:.0f}} mm<br>masa=%{{y:,.0f}} g<br>isla=%{{customdata[0]}}<br>leverage=%{{customdata[3]:.4f}}<extra></extra>"))
fig.add_trace(go.Scatter(x=linea_x, y=beta[0]+beta[1]*linea_x, mode="lines", name="Ajuste ilustrativo", line={"color": "#173F5F", "width": 2}, hovertemplate="ajuste simple<extra></extra>"))
raros = validos[validos["outlier_iqr"]]
fig.add_trace(go.Scatter(x=raros["flipper_length_mm"], y=raros["body_mass_g"], mode="markers", name="Outlier IQR", marker={"color": "#D99B2B", "size": 15, "symbol": "diamond-open", "line": {"color": "#173F5F", "width": 2}}, customdata=raros[["species", "island", "leverage"]], hovertemplate="%{customdata[0]}<br>masa=%{y:,.0f} g<br>aleta=%{x:.0f} mm<br>outlier dentro de especie<extra></extra>"))
influyentes = validos[alto]
fig.add_trace(go.Scatter(x=influyentes["flipper_length_mm"], y=influyentes["body_mass_g"], mode="markers", name="Alto leverage", marker={"color": "#6E5AA8", "size": 13, "symbol": "star-open", "line": {"width": 2}}, customdata=influyentes[["species", "leverage"]], hovertemplate="%{customdata[0]}<br>masa=%{y:,.0f} g<br>aleta=%{x:.0f} mm<br>leverage=%{customdata[1]:.4f}<extra></extra>"))
errores_plot = validos[validos["error_captura"]]
fig.add_trace(go.Scatter(x=errores_plot["flipper_length_mm"], y=errores_plot["body_mass_g"], mode="markers", name="Posible error", marker={"color": "#C86B3C", "size": 16, "symbol": "x", "line": {"width": 2}}, hovertemplate="revisar captura<extra></extra>"))
n_grupos = len(grupos_validos); total = n_grupos + 4
todos = [True] * total; solo_contexto = [True] * n_grupos + [True, False, False, False]; solo_outlier = [False] * (n_grupos+1) + [True, False, False]; solo_leverage = [False] * (n_grupos+1) + [False, True, False]; solo_error = [False] * (n_grupos+1) + [False, False, True]
errores_n = int(analisis["error_captura"].sum())
if errores_n:
    fig.add_annotation(x=.01, y=.99, xref="paper", yref="paper", xanchor="left", yanchor="top", text=f"{errores_n} registros incompletos/fuera de rango no aparecen completos en el plano", showarrow=False, bgcolor="#FFF5E6", bordercolor="#D99B2B")
subtitulo = f"{especie} · n={len(validos)} completos · IQR por especie={int(validos['outlier_iqr'].sum())} · leverage>{umbral_h:.4f}: {int(alto.sum())} · captura={errores_n}"
fig.update_layout(template="plotly_white", height=560, title=f"Masa corporal frente a longitud de aleta<br><sup>{subtitulo} · leverage de un ajuste lineal ilustrativo</sup>", xaxis_title="Longitud de aleta (mm)", yaxis_title="Masa corporal (g)", legend={"orientation": "h", "y": -0.22}, margin={"t": 120, "b": 115}, font={"family": "Arial", "color": "#173F5F"}, hoverlabel={"bgcolor": "#FFFFFF"}, updatemenus=[{"x": 1, "y": 1.2, "xanchor": "right", "buttons": [{"label": "Todos", "method": "update", "args": [{"visible": todos}]}, {"label": "Contexto", "method": "update", "args": [{"visible": solo_contexto}]}, {"label": "Outliers IQR", "method": "update", "args": [{"visible": solo_outlier}]}, {"label": "Alto leverage", "method": "update", "args": [{"visible": solo_leverage}]}, {"label": "Captura", "method": "update", "args": [{"visible": solo_error}]}]}])
fig.show()


display(Markdown("**Lo que muestra:** Outlier significa extremo dentro de su especie; leverage significa posición extrema en la variable explicativa del ajuste. Dos filas incompletas requieren revisar captura. Los dos outliers completos siguen dentro de rangos plausibles: son casos raros por verificar, no errores para borrar."))

**Lo que muestra:** Outlier significa extremo dentro de su especie; leverage significa posición extrema en la variable explicativa del ajuste. Dos filas incompletas requieren revisar captura. Los dos outliers completos siguen dentro de rangos plausibles: son casos raros por verificar, no errores para borrar.

## Cómo se conecta todo

El resumen ubica centro y dispersión; la distribución revela forma; la comparación visual decide qué diferencias quedan visibles; y la revisión separa señales estadísticas de problemas de captura. Cambiar la gráfica o marcar un punto no cambia el registro: cambia la pregunta que podemos contestar.

## Decisión

El cartel usará un violín por especie con mediana visible; la ECDF quedará para consultar percentiles. Los casos raros y registros incompletos irán a revisión, nunca a borrado automático.

**Regla:** Elige la gráfica por la pregunta y verifica el contexto antes de convertir un punto raro en dato malo.